In [13]:
!pip install -q -U "transformers>=4.37.0" accelerate

## import Required libraries

In [14]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

##  Load Pre-trained Transformer Model and Tokenizer

In [15]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

print("Model loaded successfully!")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded successfully!


## Define Chatbot Response Function
This function:

Takes user input
Adds it to conversation history
Generates a response using the transformer model
Returns the chatbot reply along with updated conversation history

In [16]:
def generate_response(messages):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    return response

## Build Interactive Chatbot Loop
The chatbot keeps running until the user types exit or quit.

In [17]:
print("Chatbot: Hello! I am your AI assistant. Type 'exit' or 'quit' to stop.")

messages = [
    {"role": "system", "content": "You are a helpful and friendly AI assistant."}
]

while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye! Have a great day.")
        break

    messages.append({"role": "user", "content": user_input})

    response = generate_response(messages)

    print("Chatbot:", response)

    messages.append({"role": "assistant", "content": response})


Chatbot: Hello! I am your AI assistant. Type 'exit' or 'quit' to stop.
You: Hello!
Chatbot: Hello! How can I assist you today?
You: What is AI?
Chatbot: AI stands for "Artificial Intelligence," which refers to the simulation of human intelligence in machines that are programmed to think and process information like humans do. AI involves developing algorithms, machine learning, deep learning, natural language processing, computer vision, robotics, etc., to enable machines to perform tasks that typically require human intelligence such as recognizing patterns, understanding language, decision-making, problem-solving, perception, reasoning
You: exit
Chatbot: Goodbye! Have a great day.
